# Track 2 — Lightweight vs Heavyweight Benchmark
Benchmarks all 8 backbones: params, FLOPs, latency, AUC.
Set `SKIP_TRAINING = False` to run actual training (needs GPU).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess, os, sys
REPO = '/content/lung-nodule-fusion-xai'
if not os.path.exists(REPO):
    subprocess.run(['git','clone','https://github.com/YOUR_USERNAME/lung-nodule-fusion-xai.git',REPO],check=True)
os.chdir(REPO); sys.path.insert(0, REPO)


In [ ]:
!pip install -q ptflops==0.7.3


In [ ]:
DRIVE_BASE    = '/content/drive/MyDrive/Kanker Kanker apa yg Kanker'
PROCESSED_DIR = '/content/drive/MyDrive/lung_nodule_processed'
RESULTS_DIR   = '/content/drive/MyDrive/results'
CKPT_DIR      = '/content/drive/MyDrive/checkpoints'
SKIP_TRAINING = True   # set False to run full 5-fold CV
import os; os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
# All 8 backbones; vgg16/vit_b_16 need 224px input
BACKBONES = [
    ('mobilenet_v3_small', 64),
    ('mobilenet_v3_large', 64),
    ('efficientnet_b0',    64),
    ('densenet121',        64),
    ('convnext_tiny',      64),
    ('resnet50',           64),
    ('vgg16',             224),
    ('vit_b_16',          224),
]


In [ ]:
# Measure params / FLOPs / latency (no training needed)
import torch
from src.models.backbones import BackboneClassifier
from src.evaluation.efficiency import count_params, measure_flops, measure_latency

eff = {}
for name, res in BACKBONES:
    model = BackboneClassifier(name, n_input_channels=3, pretrained=False)
    n = count_params(model)
    gflops, _ = measure_flops(model, input_res=(3, res, res))
    lat = measure_latency(model, input_res=(3, res, res), n=20)
    eff[name] = {'params_M': n/1e6, 'gflops': gflops, 'latency_ms': lat}
    print(f'{name:25s}  {n/1e6:.1f}M  {gflops:.2f}G  {lat:.1f}ms')


In [ ]:
import numpy as np
from src.evaluation.metrics import compute_metrics, bootstrap_ci

auc_results = {}

if SKIP_TRAINING:
    # Placeholder: load from saved checkpoint results or use dummy values
    # Replace with: np.load(f'{RESULTS_DIR}/{name}_fold_results.npy', allow_pickle=True)
    print('SKIP_TRAINING=True — using dummy AUC. Set False for real training.')
    rng = np.random.default_rng(42)
    for name, _ in BACKBONES:
        n = 200
        y_true = rng.integers(0, 2, n)
        y_prob = np.clip(y_true * 0.6 + rng.normal(0, 0.3, n), 0, 1)
        m = compute_metrics(y_true, y_prob)
        lo, hi = bootstrap_ci(y_true, y_prob, n_iterations=500)
        auc_results[name] = {'auc': m['auc'], 'auc_ci_low': lo, 'auc_ci_high': hi}
else:
    import pandas as pd
    from src.training.trainer import run_kfold_cv
    from src.training.dataset import NoduleDataset2_5D
    from torch.utils.data import DataLoader
    from sklearn.metrics import roc_auc_score

    labels_df = pd.read_csv(f'{PROCESSED_DIR}/labels.csv')
    for name, res in BACKBONES:
        print(f'Training {name} ...')
        def _model_factory(n=name, r=res):
            return BackboneClassifier(n, n_input_channels=3, pretrained=True)
        def _ds_factory(df, augment, r=res):
            ds = NoduleDataset2_5D(df, patch_xy=r, augment=augment)
            return DataLoader(ds, batch_size=16, shuffle=augment, num_workers=2)
        fold_results = run_kfold_cv(_model_factory, _ds_factory, labels_df,
                                     checkpoint_dir=f'{CKPT_DIR}/{name}')
        y_true = np.concatenate([r['y_true'] for r in fold_results])
        y_prob = np.concatenate([r['y_prob'] for r in fold_results])
        m = compute_metrics(y_true, y_prob)
        lo, hi = bootstrap_ci(y_true, y_prob)
        auc_results[name] = {'auc': m['auc'], 'auc_ci_low': lo, 'auc_ci_high': hi}
        np.save(f'{RESULTS_DIR}/{name}_fold_results.npy', fold_results)


In [ ]:
from src.evaluation.statistical_tests import delong_test
import numpy as np

# DeLong: lightest (MobileNetV3-Small) vs heaviest (VGG-16)
# Requires y_true/y_prob arrays — only meaningful when SKIP_TRAINING=False
if not SKIP_TRAINING:
    z, p, delta = delong_test(y_true_light, y_prob_light, y_prob_heavy)
    print(f'DeLong MobileNetV3-Small vs VGG-16: ΔAUC={delta:.4f}, p={p:.4f}')
else:
    print('Skip DeLong (no real predictions). Run with SKIP_TRAINING=False.')


In [ ]:
from src.evaluation.efficiency import build_efficiency_table, plot_params_vs_auc, plot_flops_vs_auc

# Merge efficiency + AUC results
merged = {name: {**eff[name], **auc_results[name]} for name in eff}
df = build_efficiency_table(merged)
df.to_csv(f'{RESULTS_DIR}/efficiency_table.csv', index=False)
print(df.to_string(index=False))

plot_params_vs_auc(df, out_png=f'{RESULTS_DIR}/params_vs_auc.png')
plot_flops_vs_auc(df,  out_png=f'{RESULTS_DIR}/flops_vs_auc.png')
print('Plots saved to', RESULTS_DIR)
